# V3 Modeling runner: TF-IDF baselines on verified Gold V3

Train Logistic Regression and LinearSVC baselines from Gold V3 train and select/report by validation Macro-F1. Gold test is checksum-verified but never loaded or evaluated here.

The experiment directory is write-once and bound to the exact Gold V3 manifest. Existing runs are verified before being reused. No V1/V2 artifacts are modified.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = os.environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/vmm_v3_project")
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.check_output(["git", "-C", str(PROJECT_DIR), "status", "--porcelain"], text=True)
    if dirty.strip():
        raise RuntimeError("Colab checkout has uncommitted changes; resolve them first")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError("Project directory contains non-Git files")
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[modeling]"], cwd=PROJECT_DIR, check=True)
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Missing project Python sources: {SRC_DIR}")
sys.path[:] = [part for part in sys.path if part != str(SRC_DIR)]
sys.path.insert(0, str(SRC_DIR))
prefix = "vietnamese_topic_classification"
for name in sorted([name for name in sys.modules if name == prefix or name.startswith(prefix + ".")], key=len, reverse=True):
    del sys.modules[name]
importlib.invalidate_caches()

from vietnamese_topic_classification.analysis.news_v3 import build_keyword_report, build_prompt, sample_news_records
from vietnamese_topic_classification.config import get_v3_domain_labels
from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake
from vietnamese_topic_classification.data.gold import GoldConfig, GoldReader, GoldV3Pipeline
from vietnamese_topic_classification.data.silver import SilverReader, SilverV3Pipeline

print("Source revision:", subprocess.check_output(["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
from google.colab import drive

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
print("Mounted Data Lake:", DATA_LAKE_ROOT)

In [ ]:
GOLD_V3_MANIFEST_NAME = None  # Set an exact manifest filename if more than one exists.
manifest_dir = DATA_LAKE_ROOT / "03_gold" / "v3" / "manifests"
available_manifests = sorted(path.name for path in manifest_dir.glob("gold_manifest_v3_*.json"))
print("Gold V3 manifests:", available_manifests)
if GOLD_V3_MANIFEST_NAME is None:
    if len(available_manifests) != 1:
        raise ValueError(f"Choose one exact GOLD_V3_MANIFEST_NAME; found {len(available_manifests)}")
    GOLD_V3_MANIFEST_NAME = available_manifests[0]
if Path(GOLD_V3_MANIFEST_NAME).name != GOLD_V3_MANIFEST_NAME or not GOLD_V3_MANIFEST_NAME.startswith("gold_manifest_v3_"):
    raise ValueError("Only a Gold V3 manifest filename is accepted")

gold_v3 = GoldReader(DATA_LAKE_ROOT, GOLD_V3_MANIFEST_NAME)
if gold_v3.version != "v3" or len(gold_v3.manifest["label_to_id"]) != 27:
    raise ValueError("Expected a 27-class Gold V3 manifest")
source = gold_v3.manifest.get("source", {})
if source.get("silver_version") != "v3" or not source.get("silver_v2_sha256"):
    raise ValueError("Gold V3 source lineage is incomplete")
silver_report_name = source.get("silver_quality_report")
if not isinstance(silver_report_name, str) or Path(silver_report_name).name != silver_report_name:
    raise ValueError("Gold V3 has an invalid Silver V3 report reference")
silver_v3 = SilverReader(DATA_LAKE_ROOT, silver_report_name)
verified_silver_v3 = SilverV3Pipeline.verify_reader(silver_v3)
if (verified_silver_v3["sha256"] != source.get("silver_sha256")
        or silver_v3.report_path.name != silver_report_name
        or silver_v3.report.get("source", {}).get("silver_v2_sha256") != source.get("silver_v2_sha256")):
    raise ValueError("Gold V3 lineage does not match the verified Silver V3 snapshot")
verified_gold = gold_v3.verify()
label_count = len(gold_v3.manifest["label_to_id"])
manifest_sha256 = hashlib.sha256(gold_v3.manifest_path.read_bytes()).hexdigest()
print("Verified Gold V3:", verified_gold)
print("Classes:", label_count, "| Manifest SHA256:", manifest_sha256)


In [ ]:
from vietnamese_topic_classification.training import ClassifierConfig, ExperimentConfig, FeatureConfig
from vietnamese_topic_classification.training.runner import run_experiment

SEED = 42
MAX_FEATURES = 100_000
MIN_DF = 2
OUTPUT_ROOT = DATA_LAKE_ROOT / "experiments" / "tfidf_baselines_v3"
SPECS = [
    {"slug": "tfidf_logreg", "classifier": "logistic_regression"},
    {"slug": "tfidf_linear_svc", "classifier": "linear_svc"},
]
if MAX_FEATURES < 1 or MIN_DF < 1:
    raise ValueError("MAX_FEATURES and MIN_DF must be positive")


def verify_existing_experiment(spec):
    directory = OUTPUT_ROOT / spec["name"]
    required = ("model.joblib", "config.json", "metadata.json", "validation_metrics.json")
    missing = [name for name in required if not (directory / name).is_file()]
    if missing:
        raise FileExistsError(f"Incomplete immutable experiment artifact: {directory}; missing {missing}")
    if (directory / "model.joblib").stat().st_size == 0:
        raise ValueError(f"Empty model artifact: {directory}")
    config = json.loads((directory / "config.json").read_text(encoding="utf-8"))
    metadata = json.loads((directory / "metadata.json").read_text(encoding="utf-8"))
    report = json.loads((directory / "validation_metrics.json").read_text(encoding="utf-8"))
    params = config.get("features", {}).get("params", {})
    if (
        config.get("name") != spec["name"]
        or config.get("gold_manifest_name") != GOLD_V3_MANIFEST_NAME
        or config.get("seed") != SEED
        or config.get("classifier", {}).get("kind") != spec["classifier"]
        or config.get("classifier", {}).get("params", {}).get("class_weight") is not None
        or config.get("features", {}).get("kind") != "tfidf"
        or params.get("ngram_range") != [1, 2]
        or params.get("min_df") != MIN_DF
        or params.get("max_features") != MAX_FEATURES
        or metadata.get("experiment_name") != spec["name"]
        or metadata.get("test_evaluated") is not False
        or metadata.get("lineage", {}).get("gold_manifest_name") != GOLD_V3_MANIFEST_NAME
        or metadata.get("lineage", {}).get("gold_manifest_sha256") != manifest_sha256
        or metadata.get("lineage", {}).get("gold_source") != source
        or len(report.get("per_class_f1", {})) != label_count
        or len(report.get("confusion_matrix", [])) != label_count
        or any(len(row) != label_count for row in report.get("confusion_matrix", []))
    ):
        raise ValueError(f"Existing experiment does not match the verified Gold V3/configuration: {directory}")
    return {key: report[key] for key in ("accuracy", "macro_f1", "weighted_f1")}


results = []
for spec in SPECS:
    spec["name"] = f"{spec['slug']}_v3_{manifest_sha256[:10]}"
    artifact_dir = OUTPUT_ROOT / spec["name"]
    if artifact_dir.exists():
        metrics = verify_existing_experiment(spec)
        print("VERIFIED EXISTING ? SKIP:", spec["name"], metrics)
    else:
        result = run_experiment(
            ExperimentConfig(
                name=spec["name"],
                gold_manifest_name=GOLD_V3_MANIFEST_NAME,
                seed=SEED,
                features=FeatureConfig(params={"ngram_range": (1, 2), "min_df": MIN_DF, "max_features": MAX_FEATURES}),
                classifier=ClassifierConfig(kind=spec["classifier"], params={"class_weight": None}),
            ),
            data_lake_root=DATA_LAKE_ROOT,
            output_root=OUTPUT_ROOT,
        )
        metrics = result["validation_metrics"]
        print("TRAINED:", spec["name"], metrics)
    results.append({"name": spec["name"], "classifier": spec["classifier"], **metrics})

results.sort(key=lambda row: (-row["macro_f1"], row["name"]))
print("Validation-only ranking:", results)
print("Best on validation Macro-F1:", results[0]["name"])
print("Gold test was verified from checksums, not loaded or evaluated.")


## Modeling outputs

Each run is stored below `experiments/tfidf_baselines_v3/<experiment_name>/` with the fitted pipeline, configuration, manifest lineage, and validation metrics. Experiment names include the Gold manifest checksum; do not overwrite or use Gold test to choose a model. If the taxonomy-development limitation applies, include it in any reported results.
